# Notebook 04 — ETL Code Generation with DVR Self-Correction

**Abstract:** Evaluates the Divide-Verify-Refine self-correction loop for
generated Python and SQL code. Novel metric: correction_effectiveness
= improvement in validity rate per correction attempt.

**References:**
- Zhang et al. (2024) — DVR: Divide-Verify-Refine framework
- CHESS: Talaei et al. (2024) — SQL synthesis with self-correction
- Annam (2025) — LLM-Powered ETL code generation

In [ ]:
# Install required packages into the active kernel environment (run once)
%pip install -q lxml matplotlib seaborn pandas numpy scipy pydantic requests tqdm httpx faiss-cpu sentence-transformers google-generativeai


In [ ]:
# Cell 1 — Setup
%matplotlib inline
import sys, os, json, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

def _find_research_root() -> str:
    sentinel = "generate_datasets.py"
    candidate = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.exists(os.path.join(candidate, sentinel)):
            return candidate
        parent = os.path.dirname(candidate)
        if parent == candidate:
            break
        candidate = parent
    sub = os.path.join(os.path.abspath(os.getcwd()), "research")
    if os.path.exists(os.path.join(sub, sentinel)):
        return sub
    raise FileNotFoundError(
        f"Could not locate research root (sentinel '{sentinel}' not found). "
        "Run from bi-platform/ or bi-platform/research/."
    )

RESEARCH_ROOT = _find_research_root()
os.chdir(RESEARCH_ROOT)
if RESEARCH_ROOT not in sys.path:
    sys.path.insert(0, RESEARCH_ROOT)
print(f"RESEARCH_ROOT = {RESEARCH_ROOT}")

from src.ingestion import MultiSourceIngester
from src.profiler import SchemaProfiler
from src.llm_client import MockLLMClient, LLMClient
from src.schema_mapper import SchemaMapper
from src.code_generator import ETLCodeGenerator
from src.evaluator import ETLEvaluator
from src.visualizer import ResearchVisualizer
from data.ground_truth.ground_truth import GROUND_TRUTH

random.seed(42)

ingester = MultiSourceIngester()
profiler = SchemaProfiler()
evaluator = ETLEvaluator()
viz = ResearchVisualizer()

try:
    real_client = LLMClient()
    llm_client = real_client if real_client.is_ollama_available() else MockLLMClient()
except Exception:
    llm_client = MockLLMClient()

mapper = SchemaMapper(llm_client=llm_client)
code_gen = ETLCodeGenerator(llm_client=llm_client)


In [ ]:
# Cell 2 — Get schema mappings first
gt_keys = {
    'dataset1_retail_sales': 'dataset1_retail_sales',
    'dataset2_hospital_records': 'dataset2_hospital_records',
    'dataset3_supplier_invoices': 'dataset3_supplier_invoices',
    'dataset4_ecommerce_events': 'dataset4_ecommerce_events',
}

mappings = {}
contexts = {}
for fname, df in datasets.items():
    short = fname.split('.')[0]
    gt_key = gt_keys.get(short)
    if not gt_key:
        continue
    ctx = profiler.profile(df, short)
    contexts[short] = ctx
    gt = GROUND_TRUTH[gt_key]
    result = mapper.map_schema(ctx, condition='routed', difficulty=gt['difficulty'])
    mappings[short] = result
    print(f'{short}: fact={result.fact_table}, dims={result.dimensions}')

In [ ]:
# Cell 3 — Code generation baseline (no correction, max_attempts=0)
print('=== Baseline: No Self-Correction (max_attempts=0) ===')

baseline_results = {}
for short, mapping in mappings.items():
    ctx = contexts[short]
    dvr = code_gen.generate(mapping, ctx, max_attempts=0)
    baseline_results[short] = dvr
    print(f"\n{short}:")
    print(f"  Python valid: {dvr.final_python_valid}")
    print(f"  SQL valid: {dvr.final_sql_valid}")

# Compute baseline validity rates
py_valid_baseline = sum(1 for r in baseline_results.values() if r.final_python_valid) / len(baseline_results)
sql_valid_baseline = sum(1 for r in baseline_results.values() if r.final_sql_valid) / len(baseline_results)
print(f"\nBaseline Python validity rate: {py_valid_baseline:.0%}")
print(f"Baseline SQL validity rate: {sql_valid_baseline:.0%}")

In [ ]:
# Cell 4 — DVR self-correction loop experiment
print('=== DVR Self-Correction Experiment ===')

attempt_values = [0, 1, 2, 3]
validity_by_attempts = {short: [] for short in mappings}
py_validity_by_attempts = {short: [] for short in mappings}

for max_att in attempt_values:
    print(f"\n--- max_attempts = {max_att} ---")
    for short, mapping in mappings.items():
        random.seed(42 + max_att * 100 + hash(short) % 100)
        ctx = contexts[short]
        dvr = code_gen.generate(mapping, ctx, max_attempts=max_att)
        
        sql_valid = 1.0 if dvr.final_sql_valid else 0.0
        py_valid = 1.0 if dvr.final_python_valid else 0.0
        validity_by_attempts[short].append(sql_valid)
        py_validity_by_attempts[short].append(py_valid)
        
        print(f"  {short}: SQL={dvr.final_sql_valid}, Py={dvr.final_python_valid}, "
              f"attempts_used={dvr.total_attempts}")

In [ ]:
# Cell 5 — Show concrete correction example
print('=== Concrete Correction Example ===')

# Pick a dataset that had invalid code
example_ds = list(mappings.keys())[0]
mapping = mappings[example_ds]
ctx = contexts[example_ds]

# Generate with 0 attempts (potentially invalid)
random.seed(42)
dvr_0 = code_gen.generate(mapping, ctx, max_attempts=0)
random.seed(42)
dvr_2 = code_gen.generate(mapping, ctx, max_attempts=2)

print(f"Dataset: {example_ds}")
print(f"\n--- Original SQL (attempt 0) ---")
print(dvr_0.codes[0].sql_code[:500])
print(f"Valid: {dvr_0.codes[0].sql_valid}")

if len(dvr_2.codes) > 1:
    print(f"\n--- Corrected SQL (after {dvr_2.total_attempts} attempts) ---")
    print(dvr_2.codes[-1].sql_code[:500])
    print(f"Valid: {dvr_2.codes[-1].sql_valid}")
else:
    print("\n(Code was already valid on first attempt)")

In [ ]:
# Cell 6 — Correction effectiveness by error type
print('=== Error Type Analysis ===')

all_error_types = []
for short, mapping in mappings.items():
    random.seed(42)
    dvr = code_gen.generate(mapping, contexts[short], max_attempts=3)
    all_error_types.extend(dvr.error_types_encountered)

if all_error_types:
    from collections import Counter
    error_counts = Counter(all_error_types)
    print("Error types encountered:")
    for err_type, count in error_counts.most_common():
        print(f"  {err_type}: {count}")
else:
    print("No errors encountered (all code valid on first attempt)")

In [ ]:
# Cell 7 — Plot: SQL validity rate vs correction attempts
fig = viz.plot_ablation_correction(attempt_values, validity_by_attempts)
display(fig); plt.close(fig)
print('Saved: results/figures/fig6_ablation_correction.pdf')

In [ ]:
# Cell 8 — Save metrics
os.makedirs('results/metrics', exist_ok=True)

metrics = {
    'validity_rate_no_correction': {
        'python': py_valid_baseline,
        'sql': sql_valid_baseline,
    },
    'validity_rate_with_correction': {
        short: validity_by_attempts[short][-1] for short in validity_by_attempts
    },
    'validity_by_attempts': {
        short: vals for short, vals in validity_by_attempts.items()
    },
    'py_validity_by_attempts': {
        short: vals for short, vals in py_validity_by_attempts.items()
    },
    'attempt_values': attempt_values,
    'correction_effectiveness': {
        short: (validity_by_attempts[short][-1] - validity_by_attempts[short][0])
        for short in validity_by_attempts
    },
    'errors_by_type': dict(Counter(all_error_types)) if all_error_types else {},
    'avg_attempts_to_fix': float(np.mean([
        baseline_results[s].total_attempts for s in baseline_results
    ])) if baseline_results else 0,
}

with open('results/metrics/04_code_generation.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: results/metrics/04_code_generation.json')
print('\n✓ Notebook 04 complete.')